# 01 — Three-Statement Model

This notebook demonstrates the **integrated three-statement engine** (`src/financial_model/`).

- Inputs are explicit assumptions (`ModelAssumptions`) — no hidden Excel links.
- The Income Statement, Balance Sheet, and Cash Flow Statement are built from the same object and cross-validate via `src/financial_model/checks`.
- Working capital is formulaic (DSO/DIO/DPO) and cash is the plug, so `A = L + E` holds by construction.

All logic lives in reusable modules; the notebook only **imports and demonstrates**.

## 1. Load assumptions
We start from a synthetic demo company (500 base revenue). `beginning_equity=197` is chosen so that beginning `A = L+E` exactly balances — try changing it and the constructor raises `ValueError`.

In [ ]:
from src.financial_model.assumptions import ModelAssumptions
from src.financial_model.three_statement import ThreeStatementModel
from src.financial_model.checks import run_all_checks, is_balanced
import pandas as pd

assumptions = ModelAssumptions(
    historical_revenue=500,
    revenue_growth=[0.06, 0.05, 0.04, 0.03, 0.03],
    gross_margin=0.40, opex_margin=0.20, da_pct_revenue=0.03, capex_pct_revenue=0.04,
    tax_rate=0.25, ar_days=45, inventory_days=30, ap_days=40,
    beginning_cash=80, beginning_ar=62, beginning_inventory=25, beginning_ppe=200,
    beginning_other_assets=20, beginning_ap=30, beginning_debt=150, beginning_other_liabilities=10,
    beginning_equity=197, beginning_retained_earnings=100,
    interest_rate=0.05, dividends_pct_net_income=0.20, share_count=10,
    forecast_years=5, base_year_label=2024,
)
print(assumptions.to_dict())


## 2. Build the statements
`ThreeStatementModel` computes Revenue → COGS → Gross Profit → EBITDA → EBIT → EBT → Taxes → Net Income, then rolls PP&E, debt, RE, and cash.

In [ ]:
model = ThreeStatementModel(assumptions)
print("Balanced?", is_balanced(model))
model.income_statement.round(1)


In [ ]:
model.balance_sheet.round(1)


In [ ]:
model.cash_flow.round(1)


## 3. Integrity checks
Four reconciliations: balance sheet (`A-L-E=0`), cash (`BS cash = CFS ending cash` and `ΔCash = OCF+ICF+FCF`), debt, and retained earnings.

In [ ]:
checks = run_all_checks(model)
for name, df in checks.items():
    print(f"\n== {name} ==")
    display(df)


## 4. Visualize & export
Charts are saved to `outputs/`; the model can also write a workbook for the Excel companion.

In [ ]:
from src.visualization.charts import plot_revenue_forecast, plot_ebitda_margin
plot_revenue_forecast(model.income_statement, save_path="../outputs/revenue_forecast.png")
plot_ebitda_margin(model.income_statement, save_path="../outputs/ebitda_margin.png")
model.to_excel("../outputs/three_statement.xlsx")
print("Saved: outputs/revenue_forecast.png, ebitda_margin.png, three_statement.xlsx")


## 5. Load from CSV / Excel (data layer)
The same engine can ingest `data/raw/sample_assumptions.csv` or an Excel assumptions sheet without code changes.

In [ ]:
from src.data.loader import load_assumptions_from_csv
a2 = load_assumptions_from_csv("../data/raw/sample_assumptions.csv")
m2 = ThreeStatementModel(a2)
m2.income_statement.round(1).head(3)
